In [ ]:
import xgboost as xgb 
import pandas as pd 
import sklearn as sk 
import re 
import matplotlib.pyplot as plt
import seaborn as sns 
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import KFold 
from sklearn.model_selection import GridSearchCV 
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV
from sklearn.metrics import accuracy_score 
from scipy.linalg import sqrtm
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
%matplotlib inline

In [ ]:
def read_data(file,label):
    data = pd.read_csv(f'./train_data/select_data/{file}.csv')
    data['label'] = label
    return data

In [ ]:
a10 = read_data('m_a = 10 GeV',1)
a30 = read_data('m_a = 30 GeV',1)
a50 = read_data('m_a = 50 GeV',1)
a70 = read_data('m_a = 70 GeV',1)
a85 = read_data('m_a = 85 GeV',1)
amumu = read_data('gammamumu',0)

In [ ]:
all_data = pd.concat([a10,a30,a50,a70,a85,amumu],axis = 0)
all_data = all_data.sample(frac=1).reset_index(drop=True)

In [ ]:
a10.columns,all_data.shape

In [ ]:
X = all_data[['Photon_PT_0', 'photon_energy',
      'Muon_PT_0', 'lead_muon_energy', 
      'Muon_PT_1','nextlead_muon_energy',
      'deltar_mu_0_mu_1', 'deltar_mu_1_gamma', 
        'dimuon_mass'
             ]]
y = all_data['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 创建模型

In [ ]:
clf = xgb.XGBClassifier(learning_rate=0.02, 
                        max_depth=4, 
                        objective='binary:logistic',
                        eval_metric=['logloss', 'auc', 'error'],
                        tree_method='hist',  # 使用 CPU 的直方法
                        device='cuda',       # 指定使用 GPU
                        verbosity=2)

In [ ]:
features = X
param_test={
    'n_estimators': range(260,380,4),
    'max_depth':range(10,18,2)
}
grid_search = GridSearchCV(estimator=clf, param_grid=param_test, scoring='accuracy', cv=5)
grid_search.fit(X_train,y_train)
grid_search.best_params_

In [ ]:
scores = grid_search.cv_results_['mean_test_score']  # 交叉验证的平均得分
params = grid_search.cv_results_['params']           # 所有参数组合

# 准备写入的数据列表
data = []
for param, score in zip(params, scores):
    n_estimators = param['n_estimators']
    max_depth = param['max_depth']
    data.append([n_estimators, max_depth, score])

with open('all_params_scores.txt', 'w') as f:
    f.write("n_estimators\tmax_depth\tscore\n")  # 写入标题行
    for n, d, score in data:
        f.write(f"{n}\t{d}\t{score:.4f}\n")

In [ ]:
# 读取保存的参数和得分数据
df = pd.read_csv('all_params_scores.txt', sep='\t')

# 创建透视表，得到每个 max_depth 和 n_estimators 组合的平均得分
pivot_table = df.pivot_table(index='max_depth', columns='n_estimators', values='score', aggfunc='mean')

# 提取 X, Y, Z 用于绘图
X, Y = np.meshgrid(pivot_table.columns, pivot_table.index)
Z = pivot_table.values

# 找到得分最高的点
max_score = Z.max()
max_index = np.unravel_index(Z.argmax(), Z.shape)
max_x = X[0, max_index[1]]
max_y = Y[max_index[0], 0]

# 绘制等高线图
plt.figure(figsize=(8, 6))
contour = plt.contourf(X, Y, Z, cmap='viridis', levels=15)  # 使用 viridis 颜色映射，15 个细节级别
plt.colorbar(contour)
plt.title('Model Score Contour Plot')
plt.xlabel('Number of Trees (n_estimators)')
plt.ylabel('Tree Depth (max_depth)')

# 标记得分最高的点，用红色实心圆圈表示
plt.plot(max_x, max_y, 'ro', markersize=8)  # 'ro' 表示红色实心圆，markersize 可调整大小
plt.text(max_x, max_y, f'Max Score: {max_score:.2f}', color='white', ha='right', va='bottom')

# 保存并展示图像
plt.savefig("./score.png")
plt.show()

In [ ]:
model = xgb.XGBClassifier(learning_rate = 0.02 , max_depth=18, n_estimators=300,
                        objective='binary:logistic',
                        eval_metric=['logloss','auc','error'])

In [ ]:
model.fit(X_train,y_train)

In [ ]:
model.save_model('clf_ALP_18_300')